# E6 — Semantic / Topic Stability Audit
## Repeated Top2Vec + BERTopic Sensitivity — Calib-BERT v2

E6 asks whether the **local reliability conclusions** in Calib-BERT are stable to the stochasticity and modeling choices of semantic topic discovery.

### Core design

**Topic discovery is fitted only on the 2,838-example Dreaddit development corpus.**  
The official 715-example test set is never used to create, select, merge, or tune topics.

The frozen E2 predictions are then evaluated inside semantic partitions assigned to the official test set.

### Primary E6 analysis

**Top2Vec stability**
- one fixed sentence-embedding model
- 20 predeclared UMAP random seeds
- fixed UMAP/HDBSCAN hyperparameters
- topic discovery on development text only
- official-test documents assigned to the nearest learned topic vector
- partition stability measured with pairwise **ARI** and **NMI**
- local **Brier** and **NLL** are primary local reliability endpoints
- local ECE is secondary because small-topic ECE is noisier
- seed 42 is the predeclared reference partition for topic alignment and semantic descriptions

### Alternative-method sensitivity

**BERTopic**
- same frozen sentence embeddings
- 5 predeclared UMAP seeds
- same development-only fitting rule
- official-test documents transformed after fitting
- within-method seed stability and Top2Vec-vs-BERTopic partition similarity are reported

### Frozen prediction conditions from E2

1. M0 Vanilla BERT
2. M0 + Temperature Scaling
3. M1 LS ε=0.10

### Interpretation rule

E6 is a **diagnostic stability analysis**, not a new model-selection exercise.

A semantic hotspot is considered more credible when:
- it can be aligned to the same reference topic across many Top2Vec seeds,
- its local Brier/NLL pattern remains directionally similar across seeds,
- and a related partition appears under the alternative BERTopic analysis.

Formal multiplicity-controlled statistical inference remains deferred to **E7**.


In [ ]:
# ============================================================
# 0. USER CONFIGURATION
# ============================================================
from pathlib import Path

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_ROOT = Path("/content/drive/MyDrive/CalibBERT_E6")
    except Exception as e:
        print("Google Drive mount failed:", repr(e))
        PROJECT_ROOT = Path("/content/calibbert_v2_e6")
else:
    PROJECT_ROOT = Path("/content/calibbert_v2_e6")

INPUT_DIR = Path("/content")
WORK_DIR = PROJECT_ROOT / "work"
OUT_DIR = PROJECT_ROOT / "outputs_e6"
CACHE_DIR = PROJECT_ROOT / "hf_cache"

for p in [WORK_DIR, OUT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

EXPECTED_TRAIN_N = 2838
EXPECTED_TEST_N = 715

EXPECTED_TRAIN_SHA256 = "4e791e60d3f04dd9f14abb1228b0d4f92bf628cc84097c1c173401e98d355f4e"
EXPECTED_TEST_SHA256 = "a3b6c9bf5d685f49d57e05d24266dddc52999843256b13e65c07267c543175fe"

EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

TOP2VEC_SEEDS = [
    7, 13, 21, 29, 42, 53, 67, 79, 87, 100,
    113, 127, 149, 163, 181, 199, 211, 227, 241, 257
]
TOP2VEC_REFERENCE_SEED = 42

BERTOPIC_SEEDS = [13, 21, 42, 87, 100]

UMAP_ARGS_BASE = {
    "n_neighbors": 15,
    "n_components": 5,
    "min_dist": 0.0,
    "metric": "cosine",
}

HDBSCAN_ARGS = {
    "min_cluster_size": 30,
    "metric": "euclidean",
    "cluster_selection_method": "eom",
}

TOPIC_MERGE_DELTA = 0.10
MIN_LOCAL_TEST_N = 20
TOP_WORDS_TO_SAVE = 12

PRIMARY_CONDITIONS = [
    ("M0_Vanilla_BERT", "none", "M0 Vanilla"),
    ("M0_Vanilla_BERT", "temperature", "M0 + Temperature Scaling"),
    ("M1_LS_0.10", "none", "M1 LS 0.10"),
]

print("Project root:", PROJECT_ROOT)
print("Outputs:", OUT_DIR)


## 1. Install semantic-topic dependencies

This notebook uses:
- Top2Vec
- BERTopic
- sentence-transformers
- UMAP
- HDBSCAN

Run this cell once in a fresh Colab runtime. A T4 GPU is recommended for sentence embedding, but E6 does not train BERT.


In [ ]:
import sys, subprocess, importlib.util

packages = [
    "sentence-transformers",
    "umap-learn",
    "hdbscan",
    "top2vec[sentence_encoders]",
    "bertopic",
]

missing = []
for pkg, module in [
    ("sentence-transformers", "sentence_transformers"),
    ("umap-learn", "umap"),
    ("hdbscan", "hdbscan"),
    ("top2vec[sentence_encoders]", "top2vec"),
    ("bertopic", "bertopic"),
]:
    if importlib.util.find_spec(module) is None:
        missing.append(pkg)

if missing:
    print("Installing:", missing)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *missing]
    )
else:
    print("All semantic-topic dependencies already available.")


## 2. Imports and environment audit

In [ ]:
import os
import sys
import gc
import json
import math
import random
import hashlib
import zipfile
import platform
import itertools
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

import sklearn
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    accuracy_score,
    f1_score,
    log_loss,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.optimize import linear_sum_assignment

import scipy

import sentence_transformers
from sentence_transformers import SentenceTransformer

import umap
from umap import UMAP

import hdbscan
from hdbscan import HDBSCAN

import top2vec
from top2vec import Top2Vec

import bertopic
from bertopic import BERTopic

from huggingface_hub import HfApi

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

env = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "umap_learn": getattr(umap, "__version__", "unknown"),
    "hdbscan": getattr(hdbscan, "__version__", "unknown"),
    "top2vec": getattr(top2vec, "__version__", "unknown"),
    "bertopic": getattr(bertopic, "__version__", "unknown"),
    "device": DEVICE,
}

print(json.dumps(env, indent=2))
(OUT_DIR / "e6_environment.json").write_text(json.dumps(env, indent=2))


## 3. Auto-discover and verify E6 inputs

Upload to `/content/`:

1. `dreaddit-train.csv`
2. `dreaddit-test.csv`
3. `calibbert_e2_lite.zip`
4. `e2_prediction_level_long.csv`

The raw Dreaddit files are identified by their canonical SHA-256 hashes, so `(1)` suffixes are acceptable.


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def find_by_hash(directory, expected_hash, suffix=".csv"):
    for p in Path(directory).glob(f"*{suffix}"):
        try:
            if sha256_file(p) == expected_hash:
                return p
        except Exception:
            pass
    return None

TRAIN_PATH = find_by_hash(INPUT_DIR, EXPECTED_TRAIN_SHA256)
TEST_PATH = find_by_hash(INPUT_DIR, EXPECTED_TEST_SHA256)
LITE_ZIP = INPUT_DIR / "calibbert_e2_lite.zip"
E2_PRED_PATH = INPUT_DIR / "e2_prediction_level_long.csv"

if TRAIN_PATH is None:
    raise FileNotFoundError("Canonical Dreaddit train CSV not found in /content.")
if TEST_PATH is None:
    raise FileNotFoundError("Canonical Dreaddit test CSV not found in /content.")
if not LITE_ZIP.exists():
    raise FileNotFoundError("calibbert_e2_lite.zip not found in /content.")
if not E2_PRED_PATH.exists():
    raise FileNotFoundError("e2_prediction_level_long.csv not found in /content.")

assert sha256_file(TRAIN_PATH) == EXPECTED_TRAIN_SHA256
assert sha256_file(TEST_PATH) == EXPECTED_TEST_SHA256

print("Train :", TRAIN_PATH)
print("Test  :", TEST_PATH)
print("Lite  :", LITE_ZIP)
print("E2 pred:", E2_PRED_PATH)
print("\nCanonical raw-data hashes: PASS")


## 4. Load development/test data and frozen E2 predictions

In [ ]:
dev_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
e2_pred = pd.read_csv(E2_PRED_PATH)

assert len(dev_df) == EXPECTED_TRAIN_N
assert len(test_df) == EXPECTED_TEST_N

required_raw = {"id", "post_id", "text", "label"}
for name, df in [("development", dev_df), ("test", test_df)]:
    missing = required_raw - set(df.columns)
    if missing:
        raise KeyError(f"{name} missing columns: {sorted(missing)}")

required_pred = {
    "row_index", "id", "post_id", "true_label",
    "model", "calibrator", "p_not_stress", "p_stress",
    "predicted_label", "confidence"
}
missing = required_pred - set(e2_pred.columns)
if missing:
    raise KeyError(f"E2 prediction file missing columns: {sorted(missing)}")

primary_pred_parts = []

for model, calibrator, label in PRIMARY_CONDITIONS:
    x = e2_pred[
        (e2_pred["model"] == model)
        & (e2_pred["calibrator"] == calibrator)
    ].copy().sort_values("row_index")

    if len(x) != EXPECTED_TEST_N:
        raise RuntimeError(
            f"Expected {EXPECTED_TEST_N} test rows for {model}/{calibrator}; got {len(x)}"
        )

    assert np.array_equal(
        x["id"].astype(str).to_numpy(),
        test_df["id"].astype(str).to_numpy()
    )
    assert np.array_equal(
        x["post_id"].astype(str).to_numpy(),
        test_df["post_id"].astype(str).to_numpy()
    )
    assert np.array_equal(
        x["true_label"].astype(int).to_numpy(),
        test_df["label"].astype(int).to_numpy()
    )

    x["condition_label"] = label
    primary_pred_parts.append(x)

primary_pred = pd.concat(primary_pred_parts, ignore_index=True)

print("Frozen E2 prediction alignment: PASS")
display(
    primary_pred.groupby(["condition_label"]).size().rename("n").reset_index()
)


## 5. Verify archived development split

E6 topic discovery uses **all 2,838 development texts**, not the official test set.

The archived split file is used here only as a provenance/alignment check.


In [ ]:
LITE_DIR = WORK_DIR / "e2_lite"
LITE_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(LITE_ZIP, "r") as z:
    z.extractall(LITE_DIR)

SPLIT_PATH = LITE_DIR / "development_split_assignment.csv"
if not SPLIT_PATH.exists():
    raise FileNotFoundError("development_split_assignment.csv missing from lite ZIP.")

split = pd.read_csv(SPLIT_PATH)

check = dev_df.loc[
    split["row_index"].astype(int),
    ["id", "post_id", "label"]
].reset_index(drop=True)

assert np.array_equal(
    check["id"].astype(str).to_numpy(),
    split["id"].astype(str).to_numpy()
)
assert np.array_equal(
    check["post_id"].astype(str).to_numpy(),
    split["post_id"].astype(str).to_numpy()
)
assert np.array_equal(
    check["label"].astype(int).to_numpy(),
    split["label"].astype(int).to_numpy()
)

print("Archived development alignment: PASS")
print(split["split"].value_counts())


## 6. Resolve and lock the sentence-embedding model

The same frozen MiniLM sentence embeddings are used for:
- Top2Vec document/topic geometry
- BERTopic sensitivity analysis

This isolates topic-modeling instability from changes in the embedding model itself.


In [ ]:
api = HfApi()
info = api.model_info(EMBEDDING_MODEL_ID)
EMBEDDING_MODEL_REVISION = info.sha

embedding_lock = {
    "model_id": EMBEDDING_MODEL_ID,
    "revision_commit": EMBEDDING_MODEL_REVISION,
    "resolved_utc": datetime.now(timezone.utc).isoformat(),
}

(OUT_DIR / "E6_EMBEDDING_MODEL_LOCK.json").write_text(
    json.dumps(embedding_lock, indent=2)
)

print(json.dumps(embedding_lock, indent=2))


## 7. Write E6 analysis lock before semantic modeling

The seed sets, hyperparameters, reference seed, local-support threshold, and frozen E2 conditions are fixed here before inspecting E6 outcomes.


In [ ]:
e6_lock = {
    "analysis": "E6 Semantic Topic Stability",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "canonical_train_sha256": EXPECTED_TRAIN_SHA256,
    "canonical_test_sha256": EXPECTED_TEST_SHA256,
    "topic_fit_corpus": "all Dreaddit development texts only (n=2838)",
    "official_test_used_for_topic_fit": False,
    "official_test_n": EXPECTED_TEST_N,
    "embedding_model": embedding_lock,
    "top2vec": {
        "seeds": TOP2VEC_SEEDS,
        "reference_seed": TOP2VEC_REFERENCE_SEED,
        "n_repeated_runs": len(TOP2VEC_SEEDS),
        "umap_args_base": UMAP_ARGS_BASE,
        "hdbscan_args": HDBSCAN_ARGS,
        "topic_merge_delta": TOPIC_MERGE_DELTA,
        "test_assignment": "nearest learned Top2Vec topic vector by cosine similarity",
        "development_assignment_for_stability": "nearest learned Top2Vec topic vector by cosine similarity",
    },
    "bertopic": {
        "seeds": BERTOPIC_SEEDS,
        "n_repeated_runs": len(BERTOPIC_SEEDS),
        "umap_args_base": UMAP_ARGS_BASE,
        "hdbscan_args": HDBSCAN_ARGS,
        "test_assignment": "BERTopic.transform with frozen embeddings",
    },
    "primary_partition_stability_metrics": ["adjusted_rand_index", "normalized_mutual_information"],
    "primary_local_reliability_endpoints": ["brier", "nll"],
    "secondary_local_endpoint": "ece",
    "min_local_test_support": MIN_LOCAL_TEST_N,
    "reference_topic_alignment": (
        "Hungarian maximum-overlap matching on development-document assignments "
        "to the predeclared Top2Vec seed-42 reference partition"
    ),
    "frozen_prediction_conditions": [
        {"model": m, "calibrator": c, "label": lab}
        for m, c, lab in PRIMARY_CONDITIONS
    ],
    "formal_multiplicity_control_deferred_to": "E7",
    "test_based_model_or_topic_method_selection": False,
}

(OUT_DIR / "E6_ANALYSIS_LOCK.json").write_text(
    json.dumps(e6_lock, indent=2)
)

print(json.dumps(e6_lock, indent=2))


## 8. Load the frozen MiniLM embedder

The embedding model is pinned to the commit recorded above.

A callable wrapper is used by Top2Vec. The same model separately encodes the official test set.


In [ ]:
embedder = SentenceTransformer(
    EMBEDDING_MODEL_ID,
    revision=EMBEDDING_MODEL_REVISION,
    cache_folder=str(CACHE_DIR),
    device=DEVICE,
)

def semantic_embed(texts):
    return embedder.encode(
        list(texts),
        batch_size=64,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

dev_texts = dev_df["text"].fillna("").astype(str).tolist()
test_texts = test_df["text"].fillna("").astype(str).tolist()

TEST_EMB_PATH = WORK_DIR / "test_minilm_embeddings.npy"

if TEST_EMB_PATH.exists():
    test_embeddings = np.load(TEST_EMB_PATH)
    print("[REUSE] test embeddings")
else:
    test_embeddings = semantic_embed(test_texts)
    np.save(TEST_EMB_PATH, test_embeddings)

assert test_embeddings.shape[0] == EXPECTED_TEST_N
assert np.isfinite(test_embeddings).all()

print("Test embeddings:", test_embeddings.shape)


## 9. Create one Top2Vec model and cache its development document vectors

Top2Vec generates the document embedding space once.  
The topic-discovery stage is then recomputed 20 times with different UMAP random seeds while holding all other settings fixed.

This design targets the stochastic topic-discovery component directly rather than repeatedly changing the semantic encoder.


In [ ]:
TOP2VEC_BASE_PATH = WORK_DIR / "top2vec_base_model"
DEV_EMB_PATH = WORK_DIR / "top2vec_dev_document_vectors.npy"

# Top2Vec callable must accept a list-like object and return an embedding matrix.
def top2vec_embedding_callable(texts):
    return semantic_embed(texts)

if TOP2VEC_BASE_PATH.exists():
    print("[REUSE] loading cached Top2Vec base model")
    top2vec_model = Top2Vec.load(str(TOP2VEC_BASE_PATH))
    # A callable embedding model is not serialized by Top2Vec.
    try:
        top2vec_model.set_embedding_model(top2vec_embedding_callable)
    except Exception:
        pass
else:
    first_seed = TOP2VEC_SEEDS[0]
    init_umap_args = {
        **UMAP_ARGS_BASE,
        "random_state": first_seed,
    }

    top2vec_model = Top2Vec(
        documents=dev_texts,
        embedding_model=top2vec_embedding_callable,
        use_embedding_model_tokenizer=False,
        min_count=10,
        topic_merge_delta=TOPIC_MERGE_DELTA,
        workers=1,
        keep_documents=False,
        umap_args=init_umap_args,
        hdbscan_args=HDBSCAN_ARGS,
        verbose=True,
    )

    top2vec_model.save(str(TOP2VEC_BASE_PATH))

dev_embeddings = np.asarray(top2vec_model.document_vectors)

assert dev_embeddings.shape[0] == EXPECTED_TRAIN_N
assert np.isfinite(dev_embeddings).all()

# Normalize defensively for cosine assignment and BERTopic reuse.
dev_embeddings = normalize(dev_embeddings)
test_embeddings = normalize(test_embeddings)

np.save(DEV_EMB_PATH, dev_embeddings)

print("Development embeddings:", dev_embeddings.shape)
print("Top2Vec base model ready.")


## 10. Helper functions for Top2Vec topic assignment and topic-word capture

In [ ]:
def nearest_topic_assignment(doc_embeddings, topic_vectors):
    tv = normalize(np.asarray(topic_vectors))
    sims = cosine_similarity(doc_embeddings, tv)
    labels = sims.argmax(axis=1).astype(int)
    scores = sims.max(axis=1)
    return labels, scores

def capture_top2vec_topic_words(model, n_words=12):
    words, scores, topic_nums = model.get_topics()
    rows = []

    for word_arr, score_arr, topic_num in zip(words, scores, topic_nums):
        rows.append({
            "topic_id": int(topic_num),
            "top_words": ", ".join([str(w) for w in word_arr[:n_words]]),
            "top_word_scores": ", ".join(
                [f"{float(s):.4f}" for s in score_arr[:n_words]]
            ),
        })

    return pd.DataFrame(rows)

def topic_count_from_vectors(model):
    return int(np.asarray(model.topic_vectors).shape[0])


## 11. Run 20 repeated Top2Vec topic-discovery seeds

After each seed:
- topic vectors are captured,
- all development documents are assigned to their nearest topic vector,
- all official-test documents are assigned to their nearest topic vector,
- assignments and topic descriptions are saved immediately.

If Colab disconnects, completed seeds are reused.


In [ ]:
top2vec_run_rows = []

for seed in TOP2VEC_SEEDS:
    assign_path = OUT_DIR / f"top2vec_seed{seed}_assignments.npz"
    words_path = OUT_DIR / f"top2vec_seed{seed}_topic_words.csv"

    if assign_path.exists() and words_path.exists():
        arr = np.load(assign_path)
        dev_topics = arr["dev_topics"]
        test_topics = arr["test_topics"]
        dev_scores = arr["dev_scores"]
        test_scores = arr["test_scores"]

        print(
            f"[REUSE] Top2Vec seed={seed} | "
            f"topics={len(np.unique(dev_topics))}"
        )
    else:
        print(f"\n===== Top2Vec topic recomputation seed={seed} =====")

        umap_args = {
            **UMAP_ARGS_BASE,
            "random_state": seed,
        }

        top2vec_model.compute_topics(
            umap_args=umap_args,
            hdbscan_args=HDBSCAN_ARGS,
            topic_merge_delta=TOPIC_MERGE_DELTA,
            gpu_umap=False,
            gpu_hdbscan=False,
            index_topics=False,
        )

        topic_vectors = np.asarray(top2vec_model.topic_vectors)

        dev_topics, dev_scores = nearest_topic_assignment(
            dev_embeddings,
            topic_vectors,
        )
        test_topics, test_scores = nearest_topic_assignment(
            test_embeddings,
            topic_vectors,
        )

        topic_words = capture_top2vec_topic_words(
            top2vec_model,
            TOP_WORDS_TO_SAVE,
        )
        topic_words["seed"] = seed
        topic_words.to_csv(words_path, index=False)

        np.savez_compressed(
            assign_path,
            dev_topics=dev_topics,
            test_topics=test_topics,
            dev_scores=dev_scores,
            test_scores=test_scores,
        )

    top2vec_run_rows.append({
        "seed": seed,
        "n_topics_development": int(len(np.unique(dev_topics))),
        "n_topics_test_assigned": int(len(np.unique(test_topics))),
        "mean_dev_topic_similarity": float(np.mean(dev_scores)),
        "mean_test_topic_similarity": float(np.mean(test_scores)),
        "min_test_topic_similarity": float(np.min(test_scores)),
    })

top2vec_runs = pd.DataFrame(top2vec_run_rows)
top2vec_runs.to_csv(
    OUT_DIR / "e6_top2vec_run_summary.csv",
    index=False,
)

display(top2vec_runs)


## 12. Pairwise Top2Vec partition stability: ARI and NMI

ARI/NMI are label-invariant, so topic numbers do not need to match across seeds.

Both development and official-test assignments are evaluated.


In [ ]:
top_assignments = {}

for seed in TOP2VEC_SEEDS:
    arr = np.load(OUT_DIR / f"top2vec_seed{seed}_assignments.npz")
    top_assignments[seed] = {
        "development": arr["dev_topics"].astype(int),
        "test": arr["test_topics"].astype(int),
    }

pair_rows = []

for seed_a, seed_b in itertools.combinations(TOP2VEC_SEEDS, 2):
    for corpus in ["development", "test"]:
        a = top_assignments[seed_a][corpus]
        b = top_assignments[seed_b][corpus]

        pair_rows.append({
            "seed_a": seed_a,
            "seed_b": seed_b,
            "corpus": corpus,
            "ari": adjusted_rand_score(a, b),
            "nmi": normalized_mutual_info_score(a, b),
        })

top2vec_pairwise = pd.DataFrame(pair_rows)

top2vec_pairwise.to_csv(
    OUT_DIR / "e6_top2vec_pairwise_stability.csv",
    index=False,
)

top2vec_stability_summary = (
    top2vec_pairwise.groupby("corpus")
    .agg(
        comparisons=("ari", "size"),
        ari_mean=("ari", "mean"),
        ari_median=("ari", "median"),
        ari_min=("ari", "min"),
        ari_max=("ari", "max"),
        nmi_mean=("nmi", "mean"),
        nmi_median=("nmi", "median"),
        nmi_min=("nmi", "min"),
        nmi_max=("nmi", "max"),
    )
    .reset_index()
)

top2vec_stability_summary.to_csv(
    OUT_DIR / "e6_top2vec_stability_summary.csv",
    index=False,
)

display(top2vec_stability_summary)


## 13. Align repeated Top2Vec runs to seed-42 reference topics

Alignment uses maximum-overlap Hungarian matching on **development assignments only**.

No official-test labels, probabilities, or calibration metrics are used to align topics.


In [ ]:
reference_dev = top_assignments[TOP2VEC_REFERENCE_SEED]["development"]
reference_test = top_assignments[TOP2VEC_REFERENCE_SEED]["test"]
reference_topics = np.unique(reference_dev)

def hungarian_topic_map(reference_labels, current_labels):
    ref_ids = np.unique(reference_labels)
    cur_ids = np.unique(current_labels)

    contingency = np.zeros(
        (len(ref_ids), len(cur_ids)),
        dtype=int
    )

    for i, r in enumerate(ref_ids):
        rmask = reference_labels == r
        for j, c in enumerate(cur_ids):
            contingency[i, j] = int(
                np.sum(rmask & (current_labels == c))
            )

    row_ind, col_ind = linear_sum_assignment(
        -contingency
    )

    mapping = {
        int(cur_ids[j]): int(ref_ids[i])
        for i, j in zip(row_ind, col_ind)
    }

    matched_docs = int(
        contingency[row_ind, col_ind].sum()
    )

    return mapping, matched_docs / len(reference_labels), contingency

alignment_rows = []
aligned_test_assignments = {}

for seed in TOP2VEC_SEEDS:
    current_dev = top_assignments[seed]["development"]
    current_test = top_assignments[seed]["test"]

    mapping, dev_overlap_fraction, contingency = hungarian_topic_map(
        reference_dev,
        current_dev,
    )

    aligned_test = np.array(
        [
            mapping.get(int(t), -100000 - int(t))
            for t in current_test
        ],
        dtype=int
    )

    aligned_test_assignments[seed] = aligned_test

    matched_current_topics = len(mapping)
    alignment_rows.append({
        "seed": seed,
        "reference_seed": TOP2VEC_REFERENCE_SEED,
        "reference_topic_count": int(len(np.unique(reference_dev))),
        "current_topic_count": int(len(np.unique(current_dev))),
        "matched_topic_count": int(matched_current_topics),
        "development_overlap_fraction_after_matching": float(dev_overlap_fraction),
        "test_fraction_mapped_to_reference_topics": float(
            np.mean(np.isin(aligned_test, reference_topics))
        ),
    })

alignment_summary = pd.DataFrame(alignment_rows)
alignment_summary.to_csv(
    OUT_DIR / "e6_top2vec_reference_alignment_summary.csv",
    index=False,
)

display(alignment_summary)


## 14. Reference seed-42 topic descriptions and support

Topic words come from development-only Top2Vec seed 42.  
Official-test support is shown only after the partition is frozen.


In [ ]:
ref_words_path = OUT_DIR / f"top2vec_seed{TOP2VEC_REFERENCE_SEED}_topic_words.csv"
ref_words = pd.read_csv(ref_words_path)

ref_dev_counts = pd.Series(reference_dev).value_counts().rename("development_n")
ref_test_counts = pd.Series(reference_test).value_counts().rename("test_n")

reference_topic_table = (
    ref_words
    .merge(
        ref_dev_counts.rename_axis("topic_id").reset_index(),
        on="topic_id",
        how="left",
    )
    .merge(
        ref_test_counts.rename_axis("topic_id").reset_index(),
        on="topic_id",
        how="left",
    )
)

reference_topic_table["development_n"] = (
    reference_topic_table["development_n"]
    .fillna(0)
    .astype(int)
)
reference_topic_table["test_n"] = (
    reference_topic_table["test_n"]
    .fillna(0)
    .astype(int)
)

reference_topic_table.to_csv(
    OUT_DIR / "Table_E6_Top2Vec_Reference_Topics.csv",
    index=False,
)

display(reference_topic_table)


## 15. Local reliability metrics

Primary local endpoints:
- **Brier**
- **NLL**

Secondary:
- ECE
- accuracy/F1 for context

Local-topic rows with fewer than 20 official-test examples are retained in the raw table but flagged as below the predeclared reporting threshold.


In [ ]:
def local_ece(y_true, probs, n_bins=10):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    correct = (pred == y_true).astype(float)

    bins = min(n_bins, max(2, len(y_true) // 5))
    edges = np.linspace(0, 1, bins + 1)

    ece = 0.0
    for b in range(bins):
        lo, hi = edges[b], edges[b + 1]
        mask = (conf >= lo) & (
            (conf <= hi)
            if b == bins - 1
            else (conf < hi)
        )
        if mask.any():
            ece += mask.mean() * abs(
                correct[mask].mean()
                - conf[mask].mean()
            )
    return float(ece)

def local_metric_bundle(y, probs):
    pred = probs.argmax(axis=1)

    p1 = np.clip(probs[:, 1], 1e-12, 1 - 1e-12)
    probs_clip = np.column_stack([1 - p1, p1])

    return {
        "n": int(len(y)),
        "stress_rate": float(np.mean(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "brier": float(np.mean((p1 - y) ** 2)),
        "nll": float(log_loss(y, probs_clip, labels=[0, 1])),
        "ece": local_ece(y, probs_clip),
    }

prediction_by_condition = {}

for model, calibrator, label in PRIMARY_CONDITIONS:
    x = primary_pred[
        (primary_pred["model"] == model)
        & (primary_pred["calibrator"] == calibrator)
    ].sort_values("row_index").reset_index(drop=True)

    prediction_by_condition[label] = {
        "y": x["true_label"].to_numpy(int),
        "probs": x[["p_not_stress", "p_stress"]].to_numpy(float),
    }

local_rows = []

for seed in TOP2VEC_SEEDS:
    raw_test_topics = top_assignments[seed]["test"]
    aligned_topics = aligned_test_assignments[seed]

    for condition_label, obj in prediction_by_condition.items():
        y = obj["y"]
        probs = obj["probs"]

        for topic_id in np.unique(raw_test_topics):
            mask = raw_test_topics == topic_id
            aligned_values = aligned_topics[mask]
            aligned_ref = (
                int(aligned_values[0])
                if len(np.unique(aligned_values)) == 1
                else -999999
            )

            m = local_metric_bundle(
                y[mask],
                probs[mask],
            )

            local_rows.append({
                "method": "Top2Vec",
                "seed": seed,
                "condition": condition_label,
                "raw_topic_id": int(topic_id),
                "aligned_reference_topic": aligned_ref,
                "meets_min_test_n": bool(m["n"] >= MIN_LOCAL_TEST_N),
                **m,
            })

top2vec_local = pd.DataFrame(local_rows)

top2vec_local.to_csv(
    OUT_DIR / "e6_top2vec_local_reliability_all.csv",
    index=False,
)

display(
    top2vec_local[
        top2vec_local["seed"] == TOP2VEC_REFERENCE_SEED
    ].sort_values(
        ["condition", "brier"],
        ascending=[True, False],
    )
)


## 16. Stability of local reliability across aligned Top2Vec topics

For each seed-42 reference topic, the notebook summarizes how often it is recovered and how variable its local Brier/NLL/ECE are across the 20 repeated partitions.

Only aligned topic cells meeting the predeclared minimum test support are used in the stability summary.


In [ ]:
eligible_local = top2vec_local[
    top2vec_local["meets_min_test_n"]
    & top2vec_local["aligned_reference_topic"].isin(reference_topics)
].copy()

aligned_summary = (
    eligible_local.groupby(
        ["condition", "aligned_reference_topic"]
    )
    .agg(
        seeds_observed=("seed", "nunique"),
        mean_test_n=("n", "mean"),
        min_test_n=("n", "min"),
        max_test_n=("n", "max"),
        brier_mean=("brier", "mean"),
        brier_std=("brier", "std"),
        brier_min=("brier", "min"),
        brier_max=("brier", "max"),
        nll_mean=("nll", "mean"),
        nll_std=("nll", "std"),
        nll_min=("nll", "min"),
        nll_max=("nll", "max"),
        ece_mean=("ece", "mean"),
        ece_std=("ece", "std"),
    )
    .reset_index()
)

aligned_summary["seed_recovery_fraction"] = (
    aligned_summary["seeds_observed"]
    / len(TOP2VEC_SEEDS)
)

aligned_summary = aligned_summary.merge(
    reference_topic_table[
        ["topic_id", "top_words"]
    ].rename(
        columns={
            "topic_id": "aligned_reference_topic"
        }
    ),
    on="aligned_reference_topic",
    how="left",
)

aligned_summary.to_csv(
    OUT_DIR / "Table_E6_Aligned_Local_Reliability_Stability.csv",
    index=False,
)

display(
    aligned_summary.sort_values(
        ["condition", "brier_mean"],
        ascending=[True, False],
    )
)


## 17. Partition-level local heterogeneity summary

This avoids over-interpreting the identity of one single topic.

For each seed × prediction condition, it summarizes:
- number of reportable topics,
- between-topic standard deviation of Brier/NLL,
- maximum local Brier/NLL,
- Brier/NLL range.

These are descriptive E6 stability diagnostics.


In [ ]:
heterogeneity_rows = []

for (seed, condition), x in top2vec_local.groupby(["seed", "condition"]):
    q = x[x["meets_min_test_n"]].copy()

    if len(q) == 0:
        continue

    heterogeneity_rows.append({
        "method": "Top2Vec",
        "seed": seed,
        "condition": condition,
        "reportable_topics": int(len(q)),
        "brier_topic_std": float(q["brier"].std(ddof=0)),
        "brier_topic_range": float(q["brier"].max() - q["brier"].min()),
        "max_local_brier": float(q["brier"].max()),
        "nll_topic_std": float(q["nll"].std(ddof=0)),
        "nll_topic_range": float(q["nll"].max() - q["nll"].min()),
        "max_local_nll": float(q["nll"].max()),
        "max_local_ece": float(q["ece"].max()),
    })

top2vec_heterogeneity = pd.DataFrame(heterogeneity_rows)

top2vec_heterogeneity.to_csv(
    OUT_DIR / "e6_top2vec_local_heterogeneity_by_seed.csv",
    index=False,
)

heterogeneity_summary = (
    top2vec_heterogeneity.groupby("condition")
    .agg(
        seeds=("seed", "nunique"),
        reportable_topics_mean=("reportable_topics", "mean"),
        brier_topic_std_mean=("brier_topic_std", "mean"),
        brier_topic_std_sd=("brier_topic_std", "std"),
        brier_topic_range_mean=("brier_topic_range", "mean"),
        max_local_brier_mean=("max_local_brier", "mean"),
        max_local_brier_sd=("max_local_brier", "std"),
        nll_topic_std_mean=("nll_topic_std", "mean"),
        nll_topic_std_sd=("nll_topic_std", "std"),
        nll_topic_range_mean=("nll_topic_range", "mean"),
        max_local_nll_mean=("max_local_nll", "mean"),
        max_local_nll_sd=("max_local_nll", "std"),
    )
    .reset_index()
)

heterogeneity_summary.to_csv(
    OUT_DIR / "Table_E6_Top2Vec_Local_Heterogeneity_Summary.csv",
    index=False,
)

display(heterogeneity_summary)


## 18. BERTopic alternative-method sensitivity

BERTopic uses the **same frozen development/test MiniLM embeddings**.

Each of five predeclared UMAP seeds is fit on development text only. Official-test topic assignments are obtained with `.transform`.


In [ ]:
bertopic_run_rows = []
bertopic_assignments = {}

for seed in BERTOPIC_SEEDS:
    assign_path = OUT_DIR / f"bertopic_seed{seed}_assignments.npz"
    topic_info_path = OUT_DIR / f"bertopic_seed{seed}_topic_info.csv"

    if assign_path.exists() and topic_info_path.exists():
        arr = np.load(assign_path)
        dev_topics = arr["dev_topics"].astype(int)
        test_topics = arr["test_topics"].astype(int)

        print(
            f"[REUSE] BERTopic seed={seed} | "
            f"dev topics incl outlier={len(np.unique(dev_topics))}"
        )
    else:
        print(f"\n===== BERTopic seed={seed} =====")

        umap_model = UMAP(
            **UMAP_ARGS_BASE,
            random_state=seed,
        )

        hdbscan_model = HDBSCAN(
            **HDBSCAN_ARGS,
            prediction_data=True,
        )

        topic_model = BERTopic(
            embedding_model=None,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            calculate_probabilities=False,
            verbose=False,
        )

        dev_topics, _ = topic_model.fit_transform(
            dev_texts,
            embeddings=dev_embeddings,
        )

        test_topics, _ = topic_model.transform(
            test_texts,
            embeddings=test_embeddings,
        )

        dev_topics = np.asarray(dev_topics, dtype=int)
        test_topics = np.asarray(test_topics, dtype=int)

        topic_info = topic_model.get_topic_info()
        topic_info["seed"] = seed
        topic_info.to_csv(topic_info_path, index=False)

        np.savez_compressed(
            assign_path,
            dev_topics=dev_topics,
            test_topics=test_topics,
        )

        del topic_model, umap_model, hdbscan_model
        gc.collect()

    bertopic_assignments[seed] = {
        "development": dev_topics,
        "test": test_topics,
    }

    bertopic_run_rows.append({
        "seed": seed,
        "development_unique_labels_including_outlier": int(
            len(np.unique(dev_topics))
        ),
        "development_non_outlier_topics": int(
            len(set(dev_topics.tolist()) - {-1})
        ),
        "development_outlier_fraction": float(
            np.mean(dev_topics == -1)
        ),
        "test_unique_labels_including_outlier": int(
            len(np.unique(test_topics))
        ),
        "test_outlier_fraction": float(
            np.mean(test_topics == -1)
        ),
    })

bertopic_runs = pd.DataFrame(bertopic_run_rows)
bertopic_runs.to_csv(
    OUT_DIR / "e6_bertopic_run_summary.csv",
    index=False,
)

display(bertopic_runs)


## 19. BERTopic seed stability and Top2Vec-vs-BERTopic method sensitivity

In [ ]:
bertopic_pair_rows = []

for seed_a, seed_b in itertools.combinations(BERTOPIC_SEEDS, 2):
    for corpus in ["development", "test"]:
        a = bertopic_assignments[seed_a][corpus]
        b = bertopic_assignments[seed_b][corpus]

        bertopic_pair_rows.append({
            "seed_a": seed_a,
            "seed_b": seed_b,
            "corpus": corpus,
            "ari": adjusted_rand_score(a, b),
            "nmi": normalized_mutual_info_score(a, b),
        })

bertopic_pairwise = pd.DataFrame(bertopic_pair_rows)
bertopic_pairwise.to_csv(
    OUT_DIR / "e6_bertopic_pairwise_stability.csv",
    index=False,
)

bertopic_stability_summary = (
    bertopic_pairwise.groupby("corpus")
    .agg(
        comparisons=("ari", "size"),
        ari_mean=("ari", "mean"),
        ari_median=("ari", "median"),
        ari_min=("ari", "min"),
        ari_max=("ari", "max"),
        nmi_mean=("nmi", "mean"),
        nmi_median=("nmi", "median"),
        nmi_min=("nmi", "min"),
        nmi_max=("nmi", "max"),
    )
    .reset_index()
)

bertopic_stability_summary.to_csv(
    OUT_DIR / "e6_bertopic_stability_summary.csv",
    index=False,
)

method_rows = []

for seed in BERTOPIC_SEEDS:
    for corpus in ["development", "test"]:
        top_ref = top_assignments[TOP2VEC_REFERENCE_SEED][corpus]
        bert = bertopic_assignments[seed][corpus]

        row = {
            "top2vec_reference_seed": TOP2VEC_REFERENCE_SEED,
            "bertopic_seed": seed,
            "corpus": corpus,
            "ari_including_bertopic_outliers": adjusted_rand_score(
                top_ref, bert
            ),
            "nmi_including_bertopic_outliers": normalized_mutual_info_score(
                top_ref, bert
            ),
            "bertopic_outlier_fraction": float(np.mean(bert == -1)),
        }

        non_out = bert != -1
        if non_out.sum() >= 2 and len(np.unique(bert[non_out])) >= 2:
            row["ari_excluding_bertopic_outliers"] = adjusted_rand_score(
                top_ref[non_out],
                bert[non_out],
            )
            row["nmi_excluding_bertopic_outliers"] = normalized_mutual_info_score(
                top_ref[non_out],
                bert[non_out],
            )
        else:
            row["ari_excluding_bertopic_outliers"] = np.nan
            row["nmi_excluding_bertopic_outliers"] = np.nan

        method_rows.append(row)

method_sensitivity = pd.DataFrame(method_rows)
method_sensitivity.to_csv(
    OUT_DIR / "Table_E6_Top2Vec_vs_BERTopic_Partition_Similarity.csv",
    index=False,
)

display(bertopic_stability_summary)
display(method_sensitivity)


## 20. BERTopic local reliability sensitivity

This repeats the local reliability calculation under BERTopic partitions.

The purpose is not to force one-to-one semantic equivalence with Top2Vec, but to test whether calibration heterogeneity remains visible under a different topic-modeling pipeline.


In [ ]:
bertopic_local_rows = []

for seed in BERTOPIC_SEEDS:
    test_topics = bertopic_assignments[seed]["test"]

    for condition_label, obj in prediction_by_condition.items():
        y = obj["y"]
        probs = obj["probs"]

        for topic_id in np.unique(test_topics):
            mask = test_topics == topic_id
            m = local_metric_bundle(
                y[mask],
                probs[mask],
            )

            bertopic_local_rows.append({
                "method": "BERTopic",
                "seed": seed,
                "condition": condition_label,
                "topic_id": int(topic_id),
                "is_outlier_topic": bool(topic_id == -1),
                "meets_min_test_n": bool(m["n"] >= MIN_LOCAL_TEST_N),
                **m,
            })

bertopic_local = pd.DataFrame(bertopic_local_rows)

bertopic_local.to_csv(
    OUT_DIR / "e6_bertopic_local_reliability_all.csv",
    index=False,
)

bertopic_heterogeneity_rows = []

for (seed, condition), x in bertopic_local.groupby(["seed", "condition"]):
    q = x[
        x["meets_min_test_n"]
        & (~x["is_outlier_topic"])
    ].copy()

    if len(q) == 0:
        continue

    bertopic_heterogeneity_rows.append({
        "method": "BERTopic",
        "seed": seed,
        "condition": condition,
        "reportable_non_outlier_topics": int(len(q)),
        "brier_topic_std": float(q["brier"].std(ddof=0)),
        "brier_topic_range": float(q["brier"].max() - q["brier"].min()),
        "max_local_brier": float(q["brier"].max()),
        "nll_topic_std": float(q["nll"].std(ddof=0)),
        "nll_topic_range": float(q["nll"].max() - q["nll"].min()),
        "max_local_nll": float(q["nll"].max()),
    })

bertopic_heterogeneity = pd.DataFrame(
    bertopic_heterogeneity_rows
)

bertopic_heterogeneity.to_csv(
    OUT_DIR / "e6_bertopic_local_heterogeneity_by_seed.csv",
    index=False,
)

display(bertopic_heterogeneity)


## 21. Consolidated method-level stability table

In [ ]:
method_summary_rows = []

for corpus in ["development", "test"]:
    top = top2vec_stability_summary[
        top2vec_stability_summary["corpus"] == corpus
    ].iloc[0]

    ber = bertopic_stability_summary[
        bertopic_stability_summary["corpus"] == corpus
    ].iloc[0]

    method_summary_rows.extend([
        {
            "method": "Top2Vec",
            "corpus": corpus,
            "repeated_runs": len(TOP2VEC_SEEDS),
            "pairwise_ari_mean": top["ari_mean"],
            "pairwise_ari_median": top["ari_median"],
            "pairwise_ari_min": top["ari_min"],
            "pairwise_nmi_mean": top["nmi_mean"],
            "pairwise_nmi_median": top["nmi_median"],
            "pairwise_nmi_min": top["nmi_min"],
        },
        {
            "method": "BERTopic",
            "corpus": corpus,
            "repeated_runs": len(BERTOPIC_SEEDS),
            "pairwise_ari_mean": ber["ari_mean"],
            "pairwise_ari_median": ber["ari_median"],
            "pairwise_ari_min": ber["ari_min"],
            "pairwise_nmi_mean": ber["nmi_mean"],
            "pairwise_nmi_median": ber["nmi_median"],
            "pairwise_nmi_min": ber["nmi_min"],
        },
    ])

method_stability_table = pd.DataFrame(method_summary_rows)

method_stability_table.to_csv(
    OUT_DIR / "Table_E6_Method_Seed_Stability.csv",
    index=False,
)

display(method_stability_table)


## 22. Exploratory robust-hotspot table

This table is explicitly **diagnostic**, not confirmatory.

It flags reference topics that:
- are recovered in at least 70% of Top2Vec seeds,
- meet minimum local support,
- and show their average local Brier/NLL relative to the global E2 condition.

No p-values or inferential claims are made here.


In [ ]:
global_rows = []

for condition_label, obj in prediction_by_condition.items():
    g = local_metric_bundle(
        obj["y"],
        obj["probs"],
    )
    global_rows.append({
        "condition": condition_label,
        "global_brier": g["brier"],
        "global_nll": g["nll"],
        "global_ece": g["ece"],
    })

global_metrics = pd.DataFrame(global_rows)

hotspots = aligned_summary.merge(
    global_metrics,
    on="condition",
    how="left",
)

hotspots["mean_brier_excess_vs_global"] = (
    hotspots["brier_mean"]
    - hotspots["global_brier"]
)
hotspots["mean_nll_excess_vs_global"] = (
    hotspots["nll_mean"]
    - hotspots["global_nll"]
)

hotspots["robustly_recovered_70pct"] = (
    hotspots["seed_recovery_fraction"] >= 0.70
)

hotspots.to_csv(
    OUT_DIR / "Table_E6_Exploratory_Robust_Hotspots.csv",
    index=False,
)

display(
    hotspots[
        hotspots["robustly_recovered_70pct"]
    ].sort_values(
        ["condition", "mean_brier_excess_vs_global"],
        ascending=[True, False],
    )
)


## 23. Figures

In [ ]:
# Figure A — Pairwise seed stability
fig, ax = plt.subplots(figsize=(8.5, 5.5))

for metric, marker in [("ari", "o"), ("nmi", "x")]:
    x = top2vec_pairwise[
        top2vec_pairwise["corpus"] == "test"
    ][metric].to_numpy()

    ax.plot(
        np.arange(len(x)),
        np.sort(x),
        marker=marker,
        linestyle="none",
        label=f"Top2Vec test {metric.upper()}",
    )

ax.set_xlabel("Pairwise seed comparison (sorted)")
ax.set_ylabel("Partition similarity")
ax.set_title("E6 Top2Vec Seed Stability — Official-Test Assignments")
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()

fig.savefig(
    OUT_DIR / "Figure_E6_Top2Vec_Seed_Stability.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    OUT_DIR / "Figure_E6_Top2Vec_Seed_Stability.pdf",
    bbox_inches="tight",
)
plt.show()

# Figure B — Local Brier stability by reference topic
plot_df = aligned_summary[
    (aligned_summary["condition"] == "M0 Vanilla")
    & (aligned_summary["seed_recovery_fraction"] >= 0.70)
].sort_values("aligned_reference_topic")

if len(plot_df):
    fig, ax = plt.subplots(figsize=(9, 5.5))

    ax.errorbar(
        plot_df["aligned_reference_topic"].astype(str),
        plot_df["brier_mean"],
        yerr=plot_df["brier_std"].fillna(0),
        marker="o",
        linestyle="none",
        capsize=3,
    )

    ax.set_xlabel("Seed-42 aligned reference topic")
    ax.set_ylabel("Local Brier score: mean ± SD across seeds")
    ax.set_title("E6 Local Reliability Stability — M0 Vanilla")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()

    fig.savefig(
        OUT_DIR / "Figure_E6_Local_Brier_Stability.png",
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        OUT_DIR / "Figure_E6_Local_Brier_Stability.pdf",
        bbox_inches="tight",
    )
    plt.show()

# Figure C — Topic counts across Top2Vec seeds
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(
    top2vec_runs["seed"].astype(str),
    top2vec_runs["n_topics_development"],
    marker="o",
)
ax.set_xlabel("Top2Vec UMAP seed")
ax.set_ylabel("Number of development topics")
ax.set_title("E6 Topic-Count Sensitivity Across Top2Vec Seeds")
ax.tick_params(axis="x", rotation=60)
ax.grid(True, alpha=0.25)
fig.tight_layout()

fig.savefig(
    OUT_DIR / "Figure_E6_Top2Vec_Topic_Counts.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    OUT_DIR / "Figure_E6_Top2Vec_Topic_Counts.pdf",
    bbox_inches="tight",
)
plt.show()


## 24. Export prediction-level topic assignments for E7

The export contains:
- official-test identity,
- frozen E2 probabilities,
- seed-42 Top2Vec reference topic,
- aligned repeated-Top2Vec topic assignments,
- BERTopic seed-42 assignment.

This allows E7 to perform post-level cluster bootstrap without rerunning topic models.


In [ ]:
e7_base = test_df[
    ["id", "post_id", "label"]
].copy()

e7_base = e7_base.rename(
    columns={"label": "true_label"}
)

e7_base["top2vec_reference_seed"] = TOP2VEC_REFERENCE_SEED
e7_base["top2vec_reference_topic"] = reference_test

for seed in TOP2VEC_SEEDS:
    e7_base[f"top2vec_aligned_seed_{seed}"] = aligned_test_assignments[seed]

if TOP2VEC_REFERENCE_SEED in BERTOPIC_SEEDS:
    e7_base["bertopic_seed42_topic"] = (
        bertopic_assignments[TOP2VEC_REFERENCE_SEED]["test"]
    )

pred_wide = None

for model, calibrator, label in PRIMARY_CONDITIONS:
    x = primary_pred[
        (primary_pred["model"] == model)
        & (primary_pred["calibrator"] == calibrator)
    ].sort_values("row_index").reset_index(drop=True)

    safe_label = (
        label.lower()
        .replace(" ", "_")
        .replace("+", "plus")
        .replace(".", "_")
    )

    q = pd.DataFrame({
        "id": x["id"].to_numpy(),
        f"{safe_label}_p_stress": x["p_stress"].to_numpy(),
        f"{safe_label}_predicted_label": x["predicted_label"].to_numpy(),
        f"{safe_label}_confidence": x["confidence"].to_numpy(),
    })

    pred_wide = (
        q if pred_wide is None
        else pred_wide.merge(q, on="id", how="inner")
    )

e7_export = e7_base.merge(
    pred_wide,
    on="id",
    how="left",
)

e7_export.to_csv(
    OUT_DIR / "e6_topic_assignments_and_predictions_for_E7.csv",
    index=False,
)

print("E7 export rows:", len(e7_export))


## 25. Evidence hashes and final E6 archive

In [ ]:
def hash_output_files(directory):
    rows = []

    for p in sorted(Path(directory).iterdir()):
        if not p.is_file():
            continue

        if p.name in {
            "E6_OUTPUT_HASHES.csv",
            "E6_MANIFEST.json",
        }:
            continue

        rows.append({
            "filename": p.name,
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

    return pd.DataFrame(rows)

hashes = hash_output_files(OUT_DIR)

hashes.to_csv(
    OUT_DIR / "E6_OUTPUT_HASHES.csv",
    index=False,
)

manifest = {
    "package": "CalibBERT_v2_E6_Semantic_Topic_Stability",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "canonical_train_sha256": EXPECTED_TRAIN_SHA256,
    "canonical_test_sha256": EXPECTED_TEST_SHA256,
    "embedding_model": embedding_lock,
    "top2vec_seeds": TOP2VEC_SEEDS,
    "bertopic_seeds": BERTOPIC_SEEDS,
    "reference_seed": TOP2VEC_REFERENCE_SEED,
    "official_test_used_for_topic_fitting": False,
    "files": hashes.to_dict(orient="records"),
}

(OUT_DIR / "E6_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2)
)

ZIP_PATH = PROJECT_ROOT / "E6_RESULTS.zip"

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:
    for p in sorted(OUT_DIR.iterdir()):
        if p.is_file():
            z.write(p, arcname=p.name)

print("Final E6 archive:", ZIP_PATH)
print("Output files:", len(list(OUT_DIR.iterdir())))


## 26. E6 completion gate

E6 is ready for final audit only when:

- [ ] raw Dreaddit hashes match the frozen canonical data
- [ ] E2 prediction rows align exactly with the 715 official-test cases
- [ ] topic fitting uses development text only
- [ ] MiniLM embedding revision is locked before topic modeling
- [ ] E6 analysis lock is written before semantic outcomes are inspected
- [ ] all 20 Top2Vec seeds complete
- [ ] Top2Vec pairwise development/test ARI and NMI are exported
- [ ] seed-42 reference alignment uses development overlap only
- [ ] local Brier/NLL stability is exported for all three frozen E2 conditions
- [ ] all 5 BERTopic sensitivity seeds complete
- [ ] BERTopic seed stability is exported
- [ ] Top2Vec-vs-BERTopic partition similarity is exported
- [ ] official-test labels/predictions are never used to create or align topics
- [ ] exploratory hotspot table is clearly separated from confirmatory inference
- [ ] E7 topic/prediction export is preserved
- [ ] `E6_RESULTS.zip` is safely retained in Google Drive

### Files to send for final E6 audit

1. `E6_RESULTS.zip`
2. `Table_E6_Method_Seed_Stability.csv`
3. `Table_E6_Aligned_Local_Reliability_Stability.csv`
4. `Table_E6_Top2Vec_vs_BERTopic_Partition_Similarity.csv`
5. `Table_E6_Exploratory_Robust_Hotspots.csv`
6. `E6_ANALYSIS_LOCK.json`

### Next stage after E6

If E6 passes, proceed to **E7 — cluster-aware statistical inference**:
- post_id cluster bootstrap,
- paired uncertainty intervals,
- multiple-comparison correction,
- final integrated evidence table across E2–E6.
